# CSE z-Only Control TARGET Input Validation and Visualization

This notebook validates and visualizes the CLE/TARGET inputs generated by the CSE experiment with direct control applied exclusively to `z`. It can be run independently; no other notebook depends on variables left in memory by this notebook.


In [ ]:
import importlib
import project_paths as project_paths
project_paths = importlib.reload(project_paths)

EXPERIMENT_NAME = 'z_control_nature_qiwenevents'
CLE_TARGET_EXPERIMENT_ROOT, INPUT_DATA_DIR, FIGURES_DIR = (
    project_paths.cle_target_experiment_paths(EXPERIMENT_NAME)
)
INPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('TARGET inputs:', INPUT_DATA_DIR)
print('Validation figures:', FIGURES_DIR)


## Validate z-Only CSE TARGET Inputs

Checks that the z-only control TARGET, the exact interventions, and the CSE summary are compatible, and confirms that the direct increments applied to `x` and `y` are exactly zero.


## Load TARGET and Exact CSE Interventions

Loads `cse_target.pkl`, `cse_interventions.pkl`, and `cse_summary.pkl`. The shared function checks dimensions, controlled cycles, and the temporal alignment of the impulses.

For the flattened time index `n`, the data must satisfy:

`TARGET[n+1] = RK4_free(TARGET[n]) + intervention[n+1]`.

This update has two distinct stages. First, the unforced Lorenz-63 equations advance the state from `n` to `n+1` with RK4. Second, `intervention[n+1]` is added directly to the resulting state. The intervention is therefore an instantaneous state increment, with units of state, rather than a tendency with units of state per unit time. It does not enter the Lorenz-63 right-hand side or any of the intermediate RK4 slope evaluations. A continuous forcing formulation would instead modify the differential equation itself, for example as `dx/dt = f_L63(x) + g(t)`.

The first intervention row of every DA cycle is exactly zero because row 0 stores the state already present at the cycle boundary. Interventions associated with that cycle occupy rows `1` through `Ta-1` and are applied after their corresponding free RK4 steps. Free cycles contain zero interventions in every row.

For this z-only experiment, every intervention has the form `u = (0, 0, u_z)`: `x` and `y` receive no direct instantaneous increment. This does not mean that `x` and `y` remain unchanged. After `z` is modified, the coupled Lorenz-63 dynamics propagate its effect to `y` and then to `x` during subsequent free integration steps. Thus, *z-only* describes the directly actuated component, not the full dynamical response.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from class_cle_target import (
    load_and_validate_cse_target_inputs,
    plot_cse_target_and_interventions,
    print_cse_target_validation_summary,
)

target_data = load_and_validate_cse_target_inputs(INPUT_DATA_DIR)
print_cse_target_validation_summary(target_data)

if target_data['cse_summary'].get('control_indices') != [2]:
    raise ValueError('Expected cse_summary control_indices=[2] for z-only control.')
non_z_intervention_max = float(
    np.max(np.abs(target_data['intervention_array'][..., :2]))
)
if non_z_intervention_max != 0.0:
    raise ValueError(
        'The z-only intervention history contains a nonzero x or y increment.'
    )
print('Maximum absolute intervention in x/y:', non_z_intervention_max)
print('✓ Direct CSE increments are restricted exclusively to z.')


## Visual Check of TARGET and Intervention Magnitudes

Shows the `x` component of the TARGET trajectory, which identifies the Lorenz-63 regime; marks the start of the controlled cycles; and plots the magnitude of each discrete z-only CSE increment.


In [ ]:
fig, axes, target_overview_path = plot_cse_target_and_interventions(
    target_data,
    FIGURES_DIR,
    control_label='z',
)
plt.show()
print('Saved:', target_overview_path)
